# GW-only: can order-based Gromov–Wasserstein matching recover where A's points sit in B?

## Description

<!-- Written before the first run (2026-10-26, ~10:30). Criteria drafted by Claude (autonomous); issue #13,
design: docs/brainstorm_2026-10-23_registration.md §9–10, decision 0003. -->

### The problem, standalone

Two related tasks share the search space [0, 1]. **A** is scarce: n_A observations (x_a, y_a). **B** is dense:
N_B observations (u_b, y_b) of a *transformed* copy of A's function: f_B(u) = f_A(T(u)), with T an unknown increasing
map (in this experiment the output map h is the identity; each task is still normalized on its own window, so B's
values are an affine-rescaled copy). If T were known, every B point could be placed in A's frame at
B_inA = (T(u_b), y_b on A's scale) and used as extra data for A. The registration problem is to recover T from the
data alone.

This notebook tests one component of the planned registration model in **isolation**: the front half, which finds a
few **landmarks**, i.e. for each A point the B point(s) it corresponds to. The truth for A point a is
u*_a = T⁻¹(x_a), the location in B's frame of the B point that sits at x_a in A's frame (computed from the
privileged B_inA of the prior; an A point is *covered* if x_a lies inside T's image of B's window, about 85–100% of
them). The back half (a displacement field that spreads the landmarks to all B points and refines them) is tested
separately (notebook 04).

### Prior (data)

Phase-1 pair prior (`experimental/regpfn/prior.py`), d = 1: f_A is a random BNN draw (pfn-repro's BNN prior, evaluated
on all of R so that shifted windows stay defined); T(u) = Kumaraswamy-warp(0.5 + a (u + s − 0.5)) with shift
s ~ U[−0.3, 0.3], scale a ~ logU[0.7, 1.4], warp log α, log β ~ N(0, 0.35²). Cells: **C00** (T = id: B is A's own
function, a sanity stratum) and **C10** (T on; the question). Each task is normalized affinely onto [0.1, 0.9] over its
own window; observation noise σ ~ logU[1e-3, 3e-2]. N_B = 64 (small scale), n_A ∈ {2, 5, 10, 20}. Evaluation:
256 fixed pairs per cell (the same pair construction as notebook 04, at N_B = 64), nested in n_A.

### Components

**Order relations.** For two points i, j of the same task: s(i, j) = (sign(x_i − x_j), sign(y_i − y_j)) ∈ {−1, 0, 1}².
They are unchanged by any increasing T and any increasing output map, and do not depend on how densely a region is
sampled. Nothing metric is used.

**GW objective (semi-relaxed).** A coupling π ∈ R^{n_A × N_B}, each row a distribution over B (every A point is
matched; B's marginal is free, since B points outside A's window have no partner):
  E(π) = (1/n_A) Σ_{a, a', b, b'} |s_A(a, a') − s_B(b, b')|² π_ab π_a'b'.
It is small when pairs of matches agree in order: if a is left of and below a', then a's partner should be left of
and below a''s partner.

**Solver 1, `peyre` (no learning).** Mirror descent on the row logits, L ← L − η ∇E(π), π = row-softmax(L), η = 1,
200 iterations; this is the semi-relaxed form of entropic GW (Peyré, Cuturi & Solomon 2016; Vincent-Cuaz et al. 2022;
to verify). E is non-convex: restart 0 starts from the uniform coupling, restarts 1–7 from random logits; **K = 1**
reports the restart with the lowest energy, **best-of-8** the restart closest to the truth (an oracle choice: the
recall a K = 8 hypothesis model could reach).

**Solver 2, `unrolled` (learned GW).** The same iteration, made learnable and weight-tied (10 steps): relations are
embedded, r = φ(s) ∈ R^16 (small MLP); the message m_{b,a'} = Σ_b' r_B(b, b') π_a'b' collects B's relations to the
current partners of the other A points; the score S_ab = Σ_a' ⟨W r_A(a, a'), m_{b,a'}⟩ / (16 n_A) rewards b when its
relations to those partners agree with a's relations to those A points; update L ← L + η S − κ q_b with learned
step η and a column-mass term κ q_b (q_b = Σ_a π_ab, discourages or encourages sharing a B point).

**Solver 3, `gru` (operator on the coupling).** Same messages, but the update is a learned operator in the style of
RAFT's update block: each (a, b) entry has a GRU state, updated from [S_ab, q_b, L_ab, π_ab], and L ← L + Linear(state).
The gate lets entries stop changing once settled (RAFT: Teed & Deng 2020, arXiv 2003.12039, to verify).

**Training of the learned solvers** (isolated component test, privileged supervision): every iterate's rows are
pulled toward a narrow target around the truth, CE(t_a, π_a) with t_ab ∝ exp(−(u_b − u*_a)² / 2·0.02²), weighted
0.8^(T−t), over covered rows; 6,000 steps, batch 32, Adam (lr 1e-3, cosine), n_A ~ U{2..20}, cells C00:C10 = 1:4.
Training CE and validation metrics (C10, n_A = 5 and 10, 128 fixed pairs) are logged every 250 steps (MLflow).

**Baselines.** `identity`: the B point nearest to u = x_a (right when T = id); `value`: the B point with the nearest
y (ignores order; ambiguous on level sets, and B's values are on its own scale).

### Metrics

Per covered A point, with b̂ = argmax_b π_ab (the row's mode; rows can be bimodal):
- **err_x** = |x_a − T(u_b̂)|: the landmark error *in A's frame*, the same unit as the field-seed errors σ of
  notebook 04's E4 (σ ∈ {0.02, 0.05, 0.1, 0.2});
- err_u = |u_b̂ − u*_a| (B's frame); **mass** = π-mass within 0.05 of u*_a (confidence on the truth);
- per (cell, n_A): median err_x, the share of landmarks with err_x < 0.05, and bootstrap 95% CIs over pairs.

### What to look for

C00 is the sanity stratum (the identity baseline is exact there; GW cannot know T = id, so it shows what order alone
gives). In C10: does GW beat the identity guess, and how fast does the share of good landmarks grow with n_A?
Do the learned solvers beat Peyré? How far is K = 1 from best-of-8 (the value of multiple hypotheses)?

### GO / NO-GO (drafted by Claude)

For the best solver, C10 at n_A = 5 (covered points):
- **USABLE** (go ahead with GW seeding): ≥ 70% of landmarks with err_x < 0.05 and median err_x ≤ 0.05.
- **PARTIAL** (improve GW before composing): better than `identity` on both (share and median, CIs separated) but
  below USABLE.
- **NOT USABLE**: no better than `identity`.
Secondary: learned beats Peyré if its share is ≥ 0.10 higher at n_A = 5 and 10.

### Limits

d = 1, h = id, N_B = 64, uniform designs, one training seed, privileged supervision for the learned solvers (this
tests whether the component *can* do the job, not how it trains end to end). Order relations ignore magnitudes, so
precision is bounded by B's spacing (~0.016 in u at N_B = 64) and by noise flipping near-ties in y.


In [ ]:
from meta_pfn.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/gwonly/<sweep_id>"  # set after `just snapshot runs/gwonly/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.